In [2]:
from hirag import HiRAG, QueryParam

/workspaces/HiRAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Install and configure async support for Jupyter notebooks
import sys
import subprocess

try:
    import nest_asyncio
    print("✅ nest_asyncio already installed")
except ImportError:
    print("📦 Installing nest_asyncio...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nest_asyncio"])
    import nest_asyncio
    print("✅ nest_asyncio installed successfully")

# Apply nest_asyncio to allow async/await in Jupyter
nest_asyncio.apply()
print("🔧 Configured async support for Jupyter")

✅ nest_asyncio already installed
🔧 Configured async support for Jupyter


In [ ]:
working_dir = "../flagstar_kg_20250825_231449"

# Initialize HiRAG
graph_func = HiRAG(
    working_dir=working_dir,
    enable_hierachical_mode=True,
    enable_llm_cache=True,
    enable_naive_rag=True,
    chunk_token_size=800,  # Smaller chunks for testing
    chunk_overlap_token_size=50
)

2025-08-26 12:31:47,523 - Load KV full_docs with 2 data
2025-08-26 12:31:47,583 - Load KV text_chunks with 175 data
2025-08-26 12:31:47,864 - Load KV llm_response_cache with 1266 data
2025-08-26 12:31:47,906 - Load KV community_reports with 546 data
2025-08-26 12:31:48,227 - Loaded graph from ../flagstar_test_workspace/graph_chunk_entity_relation.graphml with 2520 nodes, 3951 edges
2025-08-26 12:31:49,448 - Load (2305, 1536) data
2025-08-26 12:31:49,456 - Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': '../flagstar_test_workspace/vdb_entities.json'} 2305 data
2025-08-26 12:31:49,512 - Load (175, 1536) data
2025-08-26 12:31:49,514 - Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': '../flagstar_test_workspace/vdb_chunks.json'} 175 data


In [9]:
working_dir_latest = "../flagstar_kg_20250825_231449"

# Initialize HiRAG
graph_func_latest = HiRAG(
    working_dir=working_dir_latest,
    enable_hierachical_mode=True,
    enable_llm_cache=True,
    enable_naive_rag=True,
    chunk_token_size=800,  # Smaller chunks for testing
    chunk_overlap_token_size=50
)

2025-08-26 12:47:19,027 - Load KV full_docs with 40 data
2025-08-26 12:47:19,032 - Load KV text_chunks with 160 data
2025-08-26 12:47:19,303 - Load KV llm_response_cache with 8858 data
2025-08-26 12:47:19,354 - Load KV community_reports with 861 data
2025-08-26 12:47:19,574 - Loaded graph from ../flagstar_kg_20250825_231449/graph_chunk_entity_relation.graphml with 4013 nodes, 7313 edges
2025-08-26 12:47:19,705 - Load (3752, 1536) data
2025-08-26 12:47:19,710 - Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': '../flagstar_kg_20250825_231449/vdb_entities.json'} 3752 data
2025-08-26 12:47:19,717 - Load (160, 1536) data
2025-08-26 12:47:19,718 - Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': '../flagstar_kg_20250825_231449/vdb_chunks.json'} 160 data


# Testing HiRAG with FlagstarBank PDFs

The log messages you see are normal - they indicate HiRAG is loading the existing knowledge graph from our previous test. The "0 data" messages mean it's initializing the storage components, but it will load the actual data when needed.

In [8]:
# Check if the knowledge graph has existing data
print("📊 Knowledge Graph Statistics:")
print(f"Working directory: {working_dir}")

# Check for existing files in the workspace
import os
if os.path.exists(working_dir):
    files = os.listdir(working_dir)
    print(f"Files in workspace: {files}")
    
    # Check if we have vector databases with data
    vdb_entities_file = os.path.join(working_dir, "vdb_entities.json")
    vdb_chunks_file = os.path.join(working_dir, "vdb_chunks.json")
    
    if os.path.exists(vdb_entities_file):
        with open(vdb_entities_file, 'r') as f:
            import json
            entities_data = json.load(f)
            print(f"📈 Entities in database: {len(entities_data.get('entities', []))}")
    
    if os.path.exists(vdb_chunks_file):
        with open(vdb_chunks_file, 'r') as f:
            chunks_data = json.load(f)
            print(f"📄 Text chunks in database: {len(chunks_data.get('entities', []))}")
            
    print("✅ Knowledge graph is ready for querying!")
else:
    print("❌ No existing knowledge graph found. You need to run the PDF processing first.")

📊 Knowledge Graph Statistics:
Working directory: ./flagstar_test_workspace
Files in workspace: []
✅ Knowledge graph is ready for querying!


In [ ]:
# # Option: Rebuild knowledge graph from PDF files
# async def rebuild_knowledge_graph():
#     """Rebuild the knowledge graph from FlagstarBank PDFs"""
#     print("🏗️  Rebuilding knowledge graph from FlagstarBank PDFs...")
    
#     try:
#         from eval.advanced_document_processor import AdvancedDocumentProcessor
#         from pathlib import Path
        
#         # Process PDFs
#         processor = AdvancedDocumentProcessor()
#         pdf_dir = Path("../lenderDocs/FlagstarBank/Flagstar_PDF")
        
#         if not pdf_dir.exists():
#             print(f"❌ PDF directory not found: {pdf_dir}")
#             print("💡 Make sure the path to FlagstarBank PDFs is correct")
#             return False
        
#         # Select a few test PDFs
#         test_pdfs = [
#             "Flagstar_ConventionalLoanCreditOverlays.pdf",
#             "Flagstar_UWConv.pdf"
#         ]
        
#         documents = []
#         for pdf_name in test_pdfs:
#             pdf_path = pdf_dir / pdf_name
#             if pdf_path.exists():
#                 print(f"📄 Processing {pdf_name}...")
#                 result = processor.process_single_pdf(pdf_path)
#                 if result:
#                     text, metadata = result
#                     documents.append(text)
#                     print(f"  ✅ Extracted {metadata.character_count:,} characters")
        
#         if documents:
#             print(f"\n🔄 Inserting {len(documents)} documents into knowledge graph...")
#             # This will use the already initialized graph_func
#             await graph_func.ainsert(documents)
#             print("✅ Knowledge graph rebuilt successfully!")
#             return True
#         else:
#             print("❌ No documents were processed successfully")
#             return False
            
#     except Exception as e:
#         print(f"❌ Error rebuilding knowledge graph: {e}")
#         return False

# # Uncomment the line below to rebuild the knowledge graph
# # await rebuild_knowledge_graph()

In [13]:
import os

os.environ["OPENAI_API_KEY"] = "APIKEY"

In [6]:
# # Quick rebuild of knowledge graph from scratch
# async def quick_rebuild():
#     """Rebuild knowledge graph using the exact same method as our successful test"""
#     print("🏗️  Quick rebuild from FlagstarBank PDFs...")
    
#     try:
#         import pdfplumber
#         from pathlib import Path
        
#         # Use the exact same PDFs that worked in our test
#         pdf_dir = Path("../lenderDocs/FlagstarBank/Flagstar_PDF")
        
#         if not pdf_dir.exists():
#             print("❌ PDF directory not found")
#             return False
        
#         test_pdfs = [
#             "Flagstar_ConventionalLoanCreditOverlays.pdf",
#             "Flagstar_UWConv.pdf"
#         ]
        
#         documents = []
#         for pdf_name in test_pdfs:
#             pdf_path = pdf_dir / pdf_name
#             if pdf_path.exists():
#                 print(f"📄 Processing {pdf_name}...")
                
#                 # Extract text using pdfplumber (same as our successful test)
#                 text = ""
#                 with pdfplumber.open(pdf_path) as pdf:
#                     for i, page in enumerate(pdf.pages):
#                         try:
#                             page_text = page.extract_text()
#                             if page_text:
#                                 text += f"\n--- Page {i+1} ---\n{page_text}\n"
#                         except Exception as e:
#                             print(f"  Warning: Error on page {i+1}: {e}")
                
#                 if text.strip():
#                     # Add document metadata (same format as our test)
#                     doc_text = f"Document: {pdf_name}\nExtracted on: 2025-08-25\n\n{text}"
#                     documents.append(doc_text)
#                     print(f"  ✅ Extracted {len(text):,} characters")
#             else:
#                 print(f"  ❌ PDF not found: {pdf_name}")
        
#         if documents:
#             print(f"\n🔄 Inserting {len(documents)} documents into knowledge graph...")
#             print("This may take a few minutes...")
            
#             # Insert documents using the same method as our test
#             await graph_func.ainsert(documents)
            
#             print("✅ Knowledge graph rebuilt successfully!")
            
#             # Verify the data was saved
#             import os
#             files = os.listdir(working_dir)
#             print(f"📁 Files now in workspace: {files}")
            
#             return True
#         else:
#             print("❌ No documents were processed successfully")
#             return False
            
#     except Exception as e:
#         print(f"❌ Error rebuilding knowledge graph: {e}")
#         import traceback
#         traceback.print_exc()
#         return False

# # Run this to rebuild:
# await quick_rebuild()

In [12]:
# Test querying the existing knowledge graph
# (No need to insert data - it's already loaded from our previous test)

print("🔍 Testing HiRAG Query on FlagstarBank Documents:")
print("=" * 60)

# Test with your original question
query = "What is the max LTV for a HELOC on a purchase transaction? And if a borrower is self employed and cant provide tax returns to prove their income, what other sort of income verification can they provide to qualify for a mortgage?"
print(f"Query: {query}")
print("-" * 60)

try:
    response = await graph_func_latest.aquery(query, param=QueryParam(mode="hi"))
    print("📝 Response:")
    print(response)
except Exception as e:
    print(f"❌ Error during query: {e}")
    print("💡 Make sure the knowledge graph was built first by running the PDF processing script.")

🔍 Testing HiRAG Query on FlagstarBank Documents:
Query: What is the max LTV for a HELOC on a purchase transaction? And if a borrower is self employed and cant provide tax returns to prove their income, what other sort of income verification can they provide to qualify for a mortgage?
------------------------------------------------------------


2025-08-26 12:51:30,025 - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-26 12:51:30,145 - Using 20 entites, 17 communities, 263 reasoning path items, 17 text units
2025-08-26 12:51:30,147 - ====== References ======:
Entities (20): ['"LOAN REQUIREMENTS"', '"MINIMUM LINE AMOUNT"', '"HELOC PRODUCT GUIDELINES"', '"TRUSTS"', '"HELOC"', '"QUALIFYING RATE"', '"$1,250"', '"4506-C TAX TRANSCRIPTS"', '"HOME EQUITY LINE OF CREDIT FAQ"', '"HOME EQUITY LINE OF CREDIT (HELOC)"', '"HOME EQUITY LINES OF CREDIT"', '"REVOCABLE TRUSTS"', '"PURCHASES"', '"MAXIMUM LTV"', '"NON-OCCUPANT CO-BORROWERS FOR LOANS WITH LTV, CLTV, OR HCLTV < 95%"', '"HTLTV"', '"QUALIFYING PAYMENT"', '"MAXIMUM LTV FOR PRIMARY RESIDENCE"', '"HOME EQUITY LINE OF CREDIT"', '"FHA, VA, OR USDA MORTGAGES"']

Communities (level, cluster_id) (17): [(0, 'Cluster 10'), (2, 'Cluster 550'), (1, 'Cluster 129'), (1, 'Cluster 128'), (0, 'Cluster 3'), (1, 'Cluster 79'), (1, 'Cluster 58'), (2, 'Cluster 372'), (0

📝 Response:
## Maximum Loan-to-Value (LTV) for HELOC in Purchase Transactions

For a Home Equity Line of Credit (HELOC) on a purchase transaction, the maximum loan-to-value (LTV) ratio is typically capped at **80%**. This means that the borrower can access up to 80% of the appraised value of their property through the HELOC for purchasing a new home. The LTV is a critical metric used by lenders to assess the risk associated with a loan. By limiting the borrowing amount to a certain percentage of the property value, lenders safeguard themselves against fluctuations in the real estate market.

Additionally, it is important to note that specific requirements can vary depending on institutional guidelines and individual borrower qualifications. If a property is classified as a primary residence, different LTV criteria may apply, particularly in cases of concurrent transactions involving both a primary mortgage and a HELOC.

## Income Verification for Self-Employed Borrowers

For self-emplo

In [ ]:
# Try some additional test queries
test_queries = [
    "What are the documentation requirements for conventional loans?",
    "What are the credit score requirements mentioned?", 
    "What are the reserve requirements for borrowers?",
    "What types of income documentation are required?"
]

print("\n🧪 Additional Test Queries:")
print("=" * 60)

for i, query in enumerate(test_queries, 1):
    print(f"\n{i}. Query: {query}")
    print("-" * 40)
    
    try:
        # Use async query
        response = await graph_func.aquery(query, param=QueryParam(mode="hi"))
        # Show a truncated response for readability
        if len(response) > 300:
            print(f"📝 Response: {response[:300]}...")
        else:
            print(f"📝 Response: {response}")
    except Exception as e:
        print(f"❌ Error: {e}")

In [ ]:
# Compare different query modes
comparison_query = "What are the main lending requirements?"

print("🔄 Comparing Query Modes:")
print("=" * 60)
print(f"Query: {comparison_query}")

modes = ["hi", "naive"]
for mode in modes:
    print(f"\n📊 Mode: {mode.upper()}")
    print("-" * 30)
    
    try:
        # Use async query
        response = await graph_func.aquery(comparison_query, param=QueryParam(mode=mode))
        # Show first 200 characters for comparison
        print(f"Response: {response[:200]}...")
    except Exception as e:
        print(f"❌ Error with {mode} mode: {e}")